# Auditoria de Outliers en raw -> processed

Este notebook implementa un panel de seguimiento para validar como estan funcionando los filtros de outliers.

Objetivos:
- Cargar los artefactos de auditoria generados por `batch_processor.py` en `Doback-Data/processed-data/outliers`.
- Inspeccionar que datos se eliminan y por que regla (`reason`).
- Visualizar outliers IMU en series temporales y puntos GPS descartados sobre mapa.
- Ver estadisticas de retencion y cobertura por cada filtro.

## Secciones
1. Import Dependencies
2. Define Configuration and Constants
3. Create Core Data Structures
4. Implement Core Functions
5. Add Input Validation and Error Handling
6. Run a Minimal End-to-End Workflow
7. Add Basic Unit Tests

## 1) Import Dependencies

Importamos librerias base y opcionales para visualizacion interactiva.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

try:
    import folium
except ImportError:
    folium = None

from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", None)

print("Dependencias cargadas")
print(f"pandas={pd.__version__} | numpy={np.__version__}")

Dependencias cargadas
pandas=2.3.3 | numpy=2.2.6


## 2) Define Configuration and Constants

Centralizamos rutas, parametros y seleccion de dataset para no tocar funciones.

In [2]:
def find_repo_root(start: Path) -> Path:
    """Find project root by looking for pyproject.toml + Doback-Data."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "Doback-Data").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
DATA_ROOT = REPO_ROOT / "Doback-Data"
PROCESSED_DIR = DATA_ROOT / "processed-data"
OUTLIERS_DIR = PROCESSED_DIR / "outliers"
STABILITY_DIR = DATA_ROOT / "Stability"

# Si queda vacio, se usara el primer source_key disponible.
SOURCE_KEY = "DOBACK024_20251001"

IMU_AXES = ["ax", "ay", "az"]
IMU_SAMPLE_COLUMNS = ["ax", "ay", "az", "gx", "gy", "gz", "accmag"]
REQUIRED_DROP_COLUMNS = ["source_key", "stage", "reason", "source_file"]

# Leyenda para explicar cada razon de descarte aplicada.
DISCARD_REASON_LEGEND = {
    "gps_invalid_lat_lon": "Coordenadas GPS invalidas o fuera de rango (lat/lon).",
    "gps_invalid_altitude": "Altitud GPS fuera de rango permitido.",
    "gps_invalid_speed": "Velocidad GPS fuera de rango permitido.",
    "gps_hdop_gt_5.0": "Calidad GPS baja: HDOP mayor que 5.0.",
    "gps_fix_lt_1": "Sin fix GPS valido (fix menor que 1).",
    "gps_satellites_lt_4": "Pocos satelites para posicion confiable (menos de 4).",
    "gps_jump_gt_100.0m": "Salto espacial entre muestras consecutivas mayor de 100 m.",
    "gps_isolated_point": "Punto aislado sin continuidad espacial/temporal con su entorno.",
    "imu_rolling_outlier": "Outlier IMU detectado por ventana rolling robusta (IQR/MAD).",
}

# Criterio de descarte esperado por regla.
DISCARD_RULE_CRITERIA = {
    "gps_invalid_lat_lon": "Se descarta si lat no esta en [-90, 90] o lon no esta en [-180, 180].",
    "gps_invalid_altitude": "Se descarta si altitud esta fuera del rango valido configurado.",
    "gps_invalid_speed": "Se descarta si velocidad esta fuera del rango valido configurado.",
    "gps_hdop_gt_5.0": "Se descarta si hdop > 5.0.",
    "gps_fix_lt_1": "Se descarta si fix < 1.",
    "gps_satellites_lt_4": "Se descarta si satellites < 4.",
    "gps_jump_gt_100.0m": "Se descarta si salto espacial consecutivo > 100 m.",
    "gps_isolated_point": "Se descarta si el punto queda aislado segun la regla de continuidad.",
    "imu_rolling_outlier": "Se descarta si ax/ay/az salen de los limites rolling robustos (IQR/MAD).",
}

# Columnas candidatas para estimar el rango observado de valores descartados.
DISCARD_REASON_VALUE_COLUMNS = {
    "gps_invalid_lat_lon": ["lat", "lon"],
    "gps_invalid_altitude": ["altitud", "alt", "altura"],
    "gps_invalid_speed": ["velocidad", "speed"],
    "gps_hdop_gt_5.0": ["hdop"],
    "gps_fix_lt_1": ["fix"],
    "gps_satellites_lt_4": ["satellites"],
    "gps_jump_gt_100.0m": ["jump_distance_m", "distance_to_prev_m", "distancia_salto_m"],
    "gps_isolated_point": ["distance_to_prev_m", "distance_to_next_m"],
    "imu_rolling_outlier": ["ax", "ay", "az"],
}

print(f"Repo root: {REPO_ROOT}")
print(f"Outliers dir: {OUTLIERS_DIR} | exists={OUTLIERS_DIR.exists()}")

Repo root: /home/acastilla/Lidar-algority/LiDAR-Stability-algorithm
Outliers dir: /home/acastilla/Lidar-algority/LiDAR-Stability-algorithm/Doback-Data/processed-data/outliers | exists=True


## 3) Create Core Data Structures

Definimos contenedores simples para pasar artefactos entre funciones.

In [3]:
@dataclass
class AuditArtifacts:
    source_key: str
    dropped_path: Path | None
    summary_path: Path | None


@dataclass
class AuditData:
    source_key: str
    dropped_df: pd.DataFrame
    summary: dict

## 4) Implement Core Functions

Funciones para descubrir artefactos, cargar datos, calcular estadisticas y visualizarlas.

In [4]:
def discover_artifacts(outliers_dir: Path) -> dict[str, AuditArtifacts]:
    """Return available outlier audit artifacts grouped by source_key."""
    if not outliers_dir.exists():
        return {}

    dropped_paths = sorted(outliers_dir.glob("*_dropped_rows.csv"))
    summary_paths = sorted(outliers_dir.glob("*_drop_summary.json"))

    summary_map = {
        p.name.replace("_drop_summary.json", ""): p
        for p in summary_paths
    }

    artifacts: dict[str, AuditArtifacts] = {}
    for dropped_path in dropped_paths:
        source_key = dropped_path.name.replace("_dropped_rows.csv", "")
        artifacts[source_key] = AuditArtifacts(
            source_key=source_key,
            dropped_path=dropped_path,
            summary_path=summary_map.get(source_key),
        )

    for source_key, summary_path in summary_map.items():
        if source_key not in artifacts:
            artifacts[source_key] = AuditArtifacts(
                source_key=source_key,
                dropped_path=None,
                summary_path=summary_path,
            )

    return artifacts


def pick_source_key(available_keys: list[str], requested_key: str | None) -> str | None:
    if not available_keys:
        return None
    if requested_key and requested_key in available_keys:
        return requested_key
    return available_keys[0]


def load_audit_data(artifacts: AuditArtifacts) -> AuditData:
    dropped_df = pd.DataFrame()
    if artifacts.dropped_path is not None and artifacts.dropped_path.exists():
        dropped_df = pd.read_csv(artifacts.dropped_path)

    summary = {}
    if artifacts.summary_path is not None and artifacts.summary_path.exists():
        summary = json.loads(artifacts.summary_path.read_text(encoding="utf-8"))

    return AuditData(source_key=artifacts.source_key, dropped_df=dropped_df, summary=summary)


def summarize_drops(dropped_df: pd.DataFrame) -> pd.DataFrame:
    if dropped_df is None or dropped_df.empty:
        return pd.DataFrame(columns=["stage", "reason", "dropped_rows", "pct_over_dropped"])

    summary = (
        dropped_df.groupby(["stage", "reason"], dropna=False)
        .size()
        .reset_index(name="dropped_rows")
        .sort_values(["stage", "dropped_rows"], ascending=[True, False])
    )
    total_dropped = summary["dropped_rows"].sum()
    summary["pct_over_dropped"] = np.where(
        total_dropped > 0,
        100.0 * summary["dropped_rows"] / total_dropped,
        np.nan,
    )
    return summary


def build_discard_reason_explanations(
    dropped_df: pd.DataFrame,
    summary: dict,
    reason_legend: dict[str, str] | None = None,
) -> pd.DataFrame:
    """Build explanation table for each discard rule that was applied."""
    columns = ["stage", "reason", "descripcion", "dropped_rows", "pct_over_dropped", "aplicada"]
    reason_legend = reason_legend or {}

    drop_summary = summarize_drops(dropped_df)

    steps = summary.get("filter_steps", []) if isinstance(summary, dict) else []
    applied_rows = []
    for step in steps:
        reason = step.get("filter_name")
        if reason is None:
            continue
        dropped_rows = step.get("dropped_rows", 0)
        try:
            dropped_rows = int(dropped_rows)
        except (TypeError, ValueError):
            dropped_rows = 0

        applied_rows.append(
            {
                "stage": step.get("stage", "unknown"),
                "reason": str(reason),
                "aplicada": True,
                "dropped_rows_step": dropped_rows,
            }
        )

    applied_df = pd.DataFrame(applied_rows)

    if drop_summary.empty and applied_df.empty:
        return pd.DataFrame(columns=columns)

    if applied_df.empty:
        base = drop_summary[["stage", "reason"]].drop_duplicates().copy()
        base["aplicada"] = True
        base["dropped_rows_step"] = np.nan
    else:
        base = applied_df.drop_duplicates(subset=["stage", "reason"], keep="first")

    if not drop_summary.empty:
        observed = drop_summary[["stage", "reason"]].drop_duplicates().copy()
        observed["aplicada"] = True
        observed["dropped_rows_step"] = np.nan
        base = pd.concat([base, observed], ignore_index=True)
        base = base.drop_duplicates(subset=["stage", "reason"], keep="first")

    result = base.merge(drop_summary, on=["stage", "reason"], how="left")
    result["dropped_rows"] = result["dropped_rows"].fillna(result["dropped_rows_step"])
    result["dropped_rows"] = result["dropped_rows"].fillna(0).astype(int)
    result["pct_over_dropped"] = result["pct_over_dropped"].fillna(0.0)
    result["descripcion"] = result["reason"].map(reason_legend).fillna(
        "Regla aplicada sin descripcion en DISCARD_REASON_LEGEND"
    )
    result["aplicada"] = result["aplicada"].fillna(True)

    result = result.sort_values(["stage", "dropped_rows", "reason"], ascending=[True, False, True])
    return result[columns]


def sample_drops_by_reason(dropped_df: pd.DataFrame, n_per_reason: int = 5) -> pd.DataFrame:
    if dropped_df is None or dropped_df.empty:
        return pd.DataFrame()
    if "reason" not in dropped_df.columns:
        return dropped_df.head(n_per_reason).copy()

    work = dropped_df.copy()
    ts_col = _pick_timestamp_column(work)
    if ts_col is not None:
        work["_ts_sort"] = pd.to_datetime(work[ts_col], errors="coerce")
        work = work.sort_values("_ts_sort", kind="stable")

    sampled = work.groupby("reason", dropna=False, group_keys=False).head(n_per_reason).copy()
    if "_ts_sort" in sampled.columns:
        sampled = sampled.drop(columns=["_ts_sort"])
    return sampled


_STABILITY_CACHE: dict[str, pd.DataFrame] = {}


def _stability_raw_path_from_source_key(source_key: str) -> Path:
    return STABILITY_DIR / f"ESTABILIDAD_{source_key}.txt"


def load_stability_reference_dataframe(source_key: str) -> pd.DataFrame:
    source_key = str(source_key)
    if source_key in _STABILITY_CACHE:
        return _STABILITY_CACHE[source_key].copy()

    path = _stability_raw_path_from_source_key(source_key)
    if not path.exists():
        _STABILITY_CACHE[source_key] = pd.DataFrame()
        return pd.DataFrame()

    try:
        from src.lidar_stability.parsers.batch_processor import parse_stability_file
    except Exception:
        _STABILITY_CACHE[source_key] = pd.DataFrame()
        return pd.DataFrame()

    ref_df = parse_stability_file(path, tracker=None, source_key=source_key, imu_filter_config={"enabled": False})
    if ref_df is None or ref_df.empty:
        ref_df = pd.DataFrame()
    else:
        ref_df = ref_df.copy()
        if "timestamp" in ref_df.columns:
            ref_df["timestamp"] = pd.to_datetime(ref_df["timestamp"], errors="coerce")
            ref_df = ref_df.dropna(subset=["timestamp"]).sort_values("timestamp", kind="stable")
        if "accmag" not in ref_df.columns and set(IMU_AXES).issubset(ref_df.columns):
            ref_df["accmag"] = np.sqrt(
                pd.to_numeric(ref_df["ax"], errors="coerce") ** 2
                + pd.to_numeric(ref_df["ay"], errors="coerce") ** 2
                + pd.to_numeric(ref_df["az"], errors="coerce") ** 2
            )

    _STABILITY_CACHE[source_key] = ref_df.copy()
    return ref_df


def enrich_drop_rows_with_stability_values(dropped_df: pd.DataFrame) -> pd.DataFrame:
    if dropped_df is None or dropped_df.empty or "source_key" not in dropped_df.columns:
        return dropped_df.copy() if dropped_df is not None else pd.DataFrame()

    enriched = dropped_df.copy()
    if "timestamp" not in enriched.columns:
        return enriched

    enriched["timestamp"] = pd.to_datetime(enriched["timestamp"], errors="coerce")
    if enriched["timestamp"].isna().all():
        return enriched

    for col in IMU_SAMPLE_COLUMNS:
        if col not in enriched.columns:
            enriched[col] = np.nan

    enriched["_row_id"] = np.arange(len(enriched))
    fillable_mask = enriched["timestamp"].notna()
    if not fillable_mask.any():
        return enriched.drop(columns=["_row_id"])

    fillable = enriched.loc[fillable_mask].copy()
    chunks = []
    for source_key, group in fillable.groupby("source_key", dropna=False):
        ref_df = load_stability_reference_dataframe(source_key)
        if ref_df.empty or "timestamp" not in ref_df.columns:
            chunks.append(group)
            continue

        merge_cols = ["timestamp"] + [col for col in IMU_SAMPLE_COLUMNS if col in ref_df.columns]
        merged = pd.merge_asof(
            group.sort_values("timestamp", kind="stable"),
            ref_df[merge_cols].sort_values("timestamp", kind="stable"),
            on="timestamp",
            direction="nearest",
            tolerance=pd.Timedelta(seconds=1),
            suffixes=("", "_ref"),
        )

        for col in IMU_SAMPLE_COLUMNS:
            ref_col = f"{col}_ref"
            if ref_col in merged.columns:
                merged[col] = merged[col].where(merged[col].notna(), merged[ref_col])
                merged = merged.drop(columns=[ref_col])

        chunks.append(merged)

    if chunks:
        filled = pd.concat(chunks, ignore_index=True)
        filled = filled.set_index("_row_id")
        enriched = enriched.set_index("_row_id")
        for col in IMU_SAMPLE_COLUMNS:
            if col in filled.columns:
                enriched.loc[filled.index, col] = filled[col]
        enriched = enriched.reset_index(drop=True)
    else:
        enriched = enriched.drop(columns=["_row_id"])

    if "accmag" in enriched.columns and set(IMU_AXES).issubset(enriched.columns):
        accmag_calc = np.sqrt(
            pd.to_numeric(enriched["ax"], errors="coerce") ** 2
            + pd.to_numeric(enriched["ay"], errors="coerce") ** 2
            + pd.to_numeric(enriched["az"], errors="coerce") ** 2
        )
        enriched["accmag"] = pd.to_numeric(enriched["accmag"], errors="coerce").where(
            pd.to_numeric(enriched["accmag"], errors="coerce").notna(),
            accmag_calc,
        )

    return enriched


def filter_steps_table(summary: dict) -> pd.DataFrame:
    steps = summary.get("filter_steps", []) if isinstance(summary, dict) else []
    if not steps:
        return pd.DataFrame(columns=["stage", "filter_name", "before_rows", "dropped_rows", "after_rows"])
    return pd.DataFrame(steps)


def _pick_timestamp_column(df: pd.DataFrame) -> str | None:
    candidates = ["timestamp", "hora_gps", "hora_raspberry"]
    for col in candidates:
        if col in df.columns:
            return col
    return None


def _resolve_imu_axes(df: pd.DataFrame) -> list[str]:
    return [axis for axis in IMU_AXES if axis in df.columns]


def plot_imu_dropped_timeline(dropped_df: pd.DataFrame, max_points: int = 25000) -> None:
    if dropped_df is None or dropped_df.empty:
        print("No hay filas descartadas para graficar.")
        return

    axes = _resolve_imu_axes(dropped_df)
    if not axes:
        print("No se encontraron columnas IMU (ax/ay/az) en el detalle de descartes.")
        return

    work = dropped_df.copy()
    if "reason" in work.columns:
        work = work[work["reason"].astype(str).str.contains("imu", case=False, na=False)].copy()
    if work.empty:
        print("No hay descartes IMU para graficar.")
        return

    ts_col = _pick_timestamp_column(work)
    if ts_col is None:
        work["_x"] = np.arange(len(work))
    else:
        work["_x"] = pd.to_datetime(work[ts_col], errors="coerce")
        if work["_x"].isna().all():
            work["_x"] = np.arange(len(work))
    work = work.sort_values("_x", kind="stable")

    if len(work) > max_points:
        work = work.sample(n=max_points, random_state=42).sort_values("_x", kind="stable")

    melted = work.melt(
        id_vars=[c for c in ["_x", "reason", "stage", "flagged_axes_count", "flagged_axes"] if c in work.columns],
        value_vars=axes,
        var_name="imu_axis",
        value_name="axis_value",
    ).dropna(subset=["axis_value"])

    if melted.empty:
        print("No hay valores IMU validos para graficar.")
        return

    fig = px.scatter(
        melted,
        x="_x",
        y="axis_value",
        color="reason" if "reason" in melted.columns else None,
        facet_row="imu_axis",
        title="Puntos IMU descartados por regla",
        opacity=0.55,
        height=900,
    )
    fig.update_traces(marker={"size": 5})
    fig.update_yaxes(matches=None)
    fig.update_layout(legend_title_text="reason")
    fig.show()


def build_dropped_points_map(dropped_df: pd.DataFrame, max_points: int = 7000):
    if dropped_df is None or dropped_df.empty:
        print("No hay datos descartados para mapear.")
        return None

    if "lat" not in dropped_df.columns or "lon" not in dropped_df.columns:
        print("No hay columnas lat/lon en descartes; no se puede construir mapa GPS.")
        return None

    gps_df = dropped_df.copy()
    if "reason" in gps_df.columns:
        gps_df = gps_df[gps_df["reason"].astype(str).str.startswith("gps_", na=False)].copy()
    if gps_df.empty:
        print("No hay descartes GPS para mapear.")
        return None

    gps_df["lat"] = pd.to_numeric(gps_df["lat"], errors="coerce")
    gps_df["lon"] = pd.to_numeric(gps_df["lon"], errors="coerce")
    gps_df = gps_df.dropna(subset=["lat", "lon"])
    gps_df = gps_df[gps_df["lat"].between(-90, 90) & gps_df["lon"].between(-180, 180)].copy()
    if gps_df.empty:
        print("No hay puntos GPS descartados con coordenadas validas.")
        return None

    if len(gps_df) > max_points:
        gps_df = gps_df.sample(n=max_points, random_state=42)

    if folium is None:
        print("Folium no esta instalado. Instala folium para visualizar mapa interactivo.")
        return gps_df

    center_lat = float(gps_df["lat"].median())
    center_lon = float(gps_df["lon"].median())
    fmap = folium.Map(location=[center_lat, center_lon], zoom_start=14, control_scale=True, tiles="CartoDB positron")

    reason_colors = {
        "gps_invalid_lat_lon": "red",
        "gps_invalid_altitude": "darkred",
        "gps_invalid_speed": "purple",
        "gps_hdop_gt_5.0": "cadetblue",
        "gps_satellites_lt_4": "orange",
        "gps_jump_gt_100.0m": "black",
        "gps_isolated_point": "pink",
    }

    for reason, reason_df in gps_df.groupby("reason", dropna=False):
        group = folium.FeatureGroup(name=f"{reason} ({len(reason_df)})", show=True)
        color = reason_colors.get(str(reason), "blue")
        for _, row in reason_df.iterrows():
            popup = f"reason={row.get('reason', '')} | stage={row.get('stage', '')}"
            folium.CircleMarker(
                location=[float(row["lat"]), float(row["lon"])],
                radius=4,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.8,
                opacity=0.9,
                popup=popup,
            ).add_to(group)
        group.add_to(fmap)

    folium.LayerControl(collapsed=False).add_to(fmap)
    return fmap

## 5) Add Input Validation and Error Handling

Guardas para columnas requeridas y validacion de paths antes de ejecutar analisis.

In [5]:
def validate_environment(outliers_dir: Path) -> None:
    if not outliers_dir.exists():
        raise FileNotFoundError(
            f"No existe directorio de outliers: {outliers_dir}. Ejecuta primero batch_processor.py"
        )


def validate_drop_columns(dropped_df: pd.DataFrame) -> list[str]:
    missing = [c for c in REQUIRED_DROP_COLUMNS if c not in dropped_df.columns]
    return missing


def safe_read_summary(path: Path | None) -> dict:
    if path is None or not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise ValueError(f"JSON invalido en summary: {path}") from exc

In [6]:
def build_unified_discard_table(
    dropped_df: pd.DataFrame,
    summary: dict,
    reason_legend: dict[str, str] | None = None,
) -> pd.DataFrame:
    """Unify explanation, stage/reason summary, and filter-steps retention in one table."""
    explanations = build_discard_reason_explanations(
        dropped_df,
        summary=summary,
        reason_legend=reason_legend,
    ).rename(columns={"dropped_rows": "dropped_rows_reason"})

    step_df = filter_steps_table(summary).copy()
    if step_df.empty:
        step_df = pd.DataFrame(
            columns=[
                "source_key",
                "stage",
                "reason",
                "before_rows",
                "dropped_rows_filter_step",
                "after_rows",
            ]
        )
    else:
        step_df = step_df.rename(
            columns={
                "filter_name": "reason",
                "dropped_rows": "dropped_rows_filter_step",
            }
        )
        for col in ["source_key", "stage", "reason", "before_rows", "dropped_rows_filter_step", "after_rows"]:
            if col not in step_df.columns:
                step_df[col] = pd.NA
        step_df = step_df[["source_key", "stage", "reason", "before_rows", "dropped_rows_filter_step", "after_rows"]]

    unified = explanations.merge(step_df, on=["stage", "reason"], how="left")

    source_key = summary.get("source_key") if isinstance(summary, dict) else None
    if "source_key" not in unified.columns:
        unified["source_key"] = source_key
    elif source_key is not None:
        unified["source_key"] = unified["source_key"].fillna(source_key)

    for col in ["before_rows", "dropped_rows_filter_step", "dropped_rows_reason", "after_rows"]:
        unified[col] = pd.to_numeric(unified[col], errors="coerce")

    unified["dropped_rows"] = (
        unified["dropped_rows_reason"]
        .fillna(unified["dropped_rows_filter_step"])
        .fillna(0)
        .astype(int)
    )
    unified["pct_over_dropped"] = pd.to_numeric(unified["pct_over_dropped"], errors="coerce").fillna(0.0).round(4)

    unified = unified.sort_values(["stage", "dropped_rows", "reason"], ascending=[True, False, True])
    return unified[
        [
            "source_key",
            "stage",
            "reason",
            "descripcion",
            "before_rows",
            "dropped_rows",
            "pct_over_dropped",
            "after_rows",
        ]
    ]


def _format_numeric_range(values: pd.Series) -> str:
    vals = pd.to_numeric(values, errors="coerce").dropna()
    if vals.empty:
        return "N/A"
    return f"[{vals.min():.4f}, {vals.max():.4f}]"


def _extract_reason_observed_range(
    dropped_df: pd.DataFrame,
    reason: str,
    reason_columns: dict[str, list[str]],
) -> str:
    if dropped_df is None or dropped_df.empty or "reason" not in dropped_df.columns:
        return "N/A"

    reason_df = dropped_df[dropped_df["reason"].astype(str) == str(reason)]
    if reason_df.empty:
        return "Sin descartes observados (0 filas)"

    candidates = reason_columns.get(str(reason), [])
    if not candidates:
        return "N/A (regla sin columna numerica directa)"

    parts = []
    for col in candidates:
        if col in reason_df.columns:
            rng = _format_numeric_range(reason_df[col])
            if rng != "N/A":
                parts.append(f"{col}={rng}")

    if not parts:
        return "N/A (sin valores numericos disponibles para esta regla)"
    return " | ".join(parts)


def build_outlier_discard_stats(
    dropped_df: pd.DataFrame,
    summary: dict,
    reason_legend: dict[str, str] | None = None,
    rule_criteria: dict[str, str] | None = None,
    reason_columns: dict[str, list[str]] | None = None,
) -> pd.DataFrame:
    """Compute discard stats by rule, including threshold criteria and observed violating ranges."""
    reason_legend = reason_legend or {}
    rule_criteria = rule_criteria or {}
    reason_columns = reason_columns or {}

    explanations = build_discard_reason_explanations(
        dropped_df,
        summary=summary,
        reason_legend=reason_legend,
    )

    if explanations.empty:
        return pd.DataFrame(
            columns=[
                "source_key",
                "stage",
                "reason",
                "descripcion",
                "criterio_descarte",
                "rango_valores_descartados",
                "n_descartes",
                "pct_over_dropped",
            ]
        )

    source_key = summary.get("source_key") if isinstance(summary, dict) else None

    stats = explanations.copy()
    stats = stats.rename(columns={"dropped_rows": "n_descartes"})
    stats["source_key"] = source_key
    stats["criterio_descarte"] = stats["reason"].map(rule_criteria).fillna("Criterio no documentado")
    stats["rango_valores_descartados"] = stats["reason"].apply(
        lambda r: _extract_reason_observed_range(dropped_df, str(r), reason_columns)
    )

    stats["n_descartes"] = pd.to_numeric(stats["n_descartes"], errors="coerce").fillna(0).astype(int)
    stats["pct_over_dropped"] = pd.to_numeric(stats["pct_over_dropped"], errors="coerce").fillna(0.0).round(4)

    stats = stats.sort_values(["stage", "n_descartes", "reason"], ascending=[True, False, True])
    return stats[
        [
            "source_key",
            "stage",
            "reason",
            "descripcion",
            "criterio_descarte",
            "rango_valores_descartados",
            "n_descartes",
            "pct_over_dropped",
        ]
    ]

## 6) Run a Minimal End-to-End Workflow

Ejecuta este bloque para cargar artefactos, mostrar estadisticas y abrir visualizaciones.

In [7]:
try:
    validate_environment(OUTLIERS_DIR)
except FileNotFoundError as exc:
    print(exc)

artifacts_map = discover_artifacts(OUTLIERS_DIR)
available_keys = sorted(artifacts_map)
print(f"Source keys disponibles: {len(available_keys)}")
print(available_keys[:20])

selected_key = pick_source_key(available_keys, SOURCE_KEY)
print(f"SOURCE_KEY seleccionado: {selected_key}")

if selected_key is None:
    print("No hay artefactos para analizar todavia.")
else:
    artifacts = artifacts_map[selected_key]
    audit_data = load_audit_data(artifacts)
    audit_data.summary = safe_read_summary(artifacts.summary_path)

    missing_columns = validate_drop_columns(audit_data.dropped_df)
    if missing_columns:
        print(f"Faltan columnas requeridas en dropped_df: {missing_columns}")

    print("\n### Tabla unificada (explicacion + resumen + retencion)")
    unified_discard_df = build_unified_discard_table(
        audit_data.dropped_df,
        summary=audit_data.summary,
        reason_legend=DISCARD_REASON_LEGEND,
    )
    display(
        unified_discard_df.style.set_properties(
            subset=["descripcion"],
            **{"white-space": "normal", "text-align": "left"},
        )
    )

    print("\n### Estadisticas de descarte por criterio/rango")
    outlier_stats_df = build_outlier_discard_stats(
        audit_data.dropped_df,
        summary=audit_data.summary,
        reason_legend=DISCARD_REASON_LEGEND,
        rule_criteria=DISCARD_RULE_CRITERIA,
        reason_columns=DISCARD_REASON_VALUE_COLUMNS,
    )
    display(
        outlier_stats_df.style.set_properties(
            subset=["descripcion", "criterio_descarte", "rango_valores_descartados"],
            **{"white-space": "normal", "text-align": "left"},
        )
    )

    print("\n### 5 filas descartadas por cada reason")
    sample_per_reason_df = sample_drops_by_reason(audit_data.dropped_df, n_per_reason=5)
    sample_per_reason_df = enrich_drop_rows_with_stability_values(sample_per_reason_df)

    # Estandariza alias de columnas para mostrar solo el set solicitado.
    projected_df = sample_per_reason_df.copy()
    alias_map = {
        "alt": ["alt", "altitud", "altura"],
        "numsats": ["numsats", "satellites"],
        "speed_kmh": ["speed_kmh", "velocidad", "speed"],
    }
    for target_col, source_candidates in alias_map.items():
        for source_col in source_candidates:
            if source_col in projected_df.columns:
                projected_df[target_col] = projected_df[source_col]
                break

    base_cols = [
        "source_key",
        "stage",
        "reason",
        "lat",
        "lon",
        "alt",
        "hdop",
        "fix",
        "numsats",
        "speed_kmh",
    ]

    stability_cols = [col for col in IMU_SAMPLE_COLUMNS if col in projected_df.columns]

    columns_to_show = [
        col for col in (base_cols + stability_cols) if col in projected_df.columns
    ]
    if columns_to_show:
        display(projected_df[columns_to_show])
    else:
        display(projected_df)

    print("\n### Mapa de puntos GPS descartados")
    dropped_map = build_dropped_points_map(audit_data.dropped_df)
    if dropped_map is not None:
        display(dropped_map)

    print("\n### Metricas globales")
    print(json.dumps({
        "source_key": audit_data.summary.get("source_key"),
        "total_dropped_rows": audit_data.summary.get("total_dropped_rows"),
        "n_filter_steps": len(audit_data.summary.get("filter_steps", [])),
    }, indent=2, ensure_ascii=False))

Source keys disponibles: 465
['DOBACK023_20250930', 'DOBACK023_20251004', 'DOBACK023_20251008', 'DOBACK023_20251010', 'DOBACK023_20251011', 'DOBACK023_20251012', 'DOBACK023_20251013', 'DOBACK023_20251016', 'DOBACK023_20251017', 'DOBACK023_20251019', 'DOBACK023_20251020', 'DOBACK023_20251022', 'DOBACK023_20251030', 'DOBACK023_20251031', 'DOBACK023_20251103', 'DOBACK023_20251106', 'DOBACK023_20251107', 'DOBACK023_20251110', 'DOBACK023_20251111', 'DOBACK023_20251112']
SOURCE_KEY seleccionado: DOBACK024_20251001

### Tabla unificada (explicacion + resumen + retencion)


,source_key,stage,reason,descripcion,before_rows,dropped_rows,pct_over_dropped,after_rows
0,DOBACK024_20251001,gps,gps_jump_gt_100.0m,Salto espacial entre muestras consecutivas mayor de 100 m.,2798,320,11.909200,2478
1,DOBACK024_20251001,gps,gps_hdop_gt_5.0,Calidad GPS baja: HDOP mayor que 5.0.,3076,259,9.639000,2817
2,DOBACK024_20251001,gps,gps_invalid_lat_lon,Coordenadas GPS invalidas o fuera de rango (lat/lon).,3269,130,4.838100,3139
3,DOBACK024_20251001,gps,gps_invalid_speed,Velocidad GPS fuera de rango permitido.,3123,47,1.749200,3076
4,DOBACK024_20251001,gps,gps_satellites_lt_4,Pocos satelites para posicion confiable (menos de 4).,2817,19,0.707100,2798
5,DOBACK024_20251001,gps,gps_isolated_point,Punto aislado sin continuidad espacial/temporal con su entorno.,2478,17,0.632700,2461
6,DOBACK024_20251001,gps,gps_invalid_altitude,Altitud GPS fuera de rango permitido.,3139,16,0.595500,3123
7,DOBACK024_20251001,gps,gps_fix_lt_1,Sin fix GPS valido (fix menor que 1).,2817,0,0.000000,2817
8,DOBACK024_20251001,stability,imu_rolling_outlier,Outlier IMU detectado por ventana rolling robusta (IQR/MAD).,91413,1879,69.929300,89534



### Estadisticas de descarte por criterio/rango


,source_key,stage,reason,descripcion,criterio_descarte,rango_valores_descartados,n_descartes,pct_over_dropped
6,DOBACK024_20251001,gps,gps_jump_gt_100.0m,Salto espacial entre muestras consecutivas mayor de 100 m.,Se descarta si salto espacial consecutivo > 100 m.,N/A (sin valores numericos disponibles para esta regla),320,11.909200
3,DOBACK024_20251001,gps,gps_hdop_gt_5.0,Calidad GPS baja: HDOP mayor que 5.0.,Se descarta si hdop > 5.0.,"hdop=[5.0100, 499.0000]",259,9.639000
0,DOBACK024_20251001,gps,gps_invalid_lat_lon,Coordenadas GPS invalidas o fuera de rango (lat/lon).,"Se descarta si lat no esta en [-90, 90] o lon no esta en [-180, 180].","lat=[0.5351, 4037144.0000] | lon=[-338652.3000, -0.5514]",130,4.838100
2,DOBACK024_20251001,gps,gps_invalid_speed,Velocidad GPS fuera de rango permitido.,Se descarta si velocidad esta fuera del rango valido configurado.,N/A (sin valores numericos disponibles para esta regla),47,1.749200
5,DOBACK024_20251001,gps,gps_satellites_lt_4,Pocos satelites para posicion confiable (menos de 4).,Se descarta si satellites < 4.,N/A (sin valores numericos disponibles para esta regla),19,0.707100
7,DOBACK024_20251001,gps,gps_isolated_point,Punto aislado sin continuidad espacial/temporal con su entorno.,Se descarta si el punto queda aislado segun la regla de continuidad.,N/A (sin valores numericos disponibles para esta regla),17,0.632700
1,DOBACK024_20251001,gps,gps_invalid_altitude,Altitud GPS fuera de rango permitido.,Se descarta si altitud esta fuera del rango valido configurado.,"alt=[6085.0000, 7093.0000]",16,0.595500
4,DOBACK024_20251001,gps,gps_fix_lt_1,Sin fix GPS valido (fix menor que 1).,Se descarta si fix < 1.,Sin descartes observados (0 filas),0,0.000000
8,DOBACK024_20251001,stability,imu_rolling_outlier,Outlier IMU detectado por ventana rolling robusta (IQR/MAD).,Se descarta si ax/ay/az salen de los limites rolling robustos (IQR/MAD).,"ax=[-280.1100, 411.1400] | ay=[-63.4400, 772.2600] | az=[728.4600, 1246.6000]",1879,69.929300



### 5 filas descartadas por cada reason


,source_key,stage,reason,lat,lon,alt,hdop,fix,numsats,speed_kmh,ax,ay,az,gx,gy,gz,accmag
0,DOBACK024_20251001,gps,gps_hdop_gt_5.0,4.053432e+01,-3.617913,715.9,6.03,1.0,5.0,0.43,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DOBACK024_20251001,gps,gps_hdop_gt_5.0,4.053441e+01,-0.551270,709.6,6.03,1.0,5.0,0.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DOBACK024_20251001,gps,gps_hdop_gt_5.0,4.053452e+01,-3.617970,701.0,6.03,1.0,5.0,0.77,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DOBACK024_20251001,gps,gps_hdop_gt_5.0,4.053455e+01,-3.617976,699.0,6.03,1.0,5.0,0.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DOBACK024_20251001,gps,gps_hdop_gt_5.0,4.053455e+01,-3.618065,699.5,6.03,1.0,5.0,0.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,DOBACK024_20251001,gps,gps_invalid_lat_lon,4.534643e+00,-3.617996,693.1,6.04,1.0,5.0,0.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,DOBACK024_20251001,gps,gps_invalid_lat_lon,4.032107e+06,-3.618154,651.3,2.91,1.0,6.0,0.34,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,DOBACK024_20251001,gps,gps_jump_gt_100.0m,4.053516e+01,-3.618158,647.8,2.92,1.0,6.0,0.62,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,DOBACK024_20251001,gps,gps_invalid_lat_lon,4.053516e+01,-337089.750000,647.9,2.92,1.0,6.0,0.36,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,DOBACK024_20251001,gps,gps_jump_gt_100.0m,4.053516e+01,-3.631547,648.2,2.92,1.0,6.0,0.36,NaN,NaN,NaN,NaN,NaN,NaN,NaN



### Mapa de puntos GPS descartados



### Metricas globales
{
  "source_key": "DOBACK024_20251001",
  "total_dropped_rows": 2687,
  "n_filter_steps": 9
}


## 7) Add Basic Unit Tests

Tests ligeros para confirmar que las funciones base se comportan como esperamos.

In [8]:
# Test 1: summarize_drops devuelve columnas esperadas cuando no hay datos
empty_summary = summarize_drops(pd.DataFrame())
assert list(empty_summary.columns) == ["stage", "reason", "dropped_rows", "pct_over_dropped"]

# Test 2: pick_source_key prioriza SOURCE_KEY si existe
assert pick_source_key(["A", "B"], "B") == "B"
assert pick_source_key(["A", "B"], "X") == "A"
assert pick_source_key([], "A") is None

# Test 3: validate_drop_columns detecta faltantes
sample = pd.DataFrame({"source_key": ["X"], "stage": ["gps"]})
missing = validate_drop_columns(sample)
assert "reason" in missing and "source_file" in missing

# Test 4: build_discard_reason_explanations incluye reglas aplicadas con y sin descartes
dropped_sample = pd.DataFrame(
    {
        "stage": ["gps", "stability"],
        "reason": ["gps_invalid_speed", "imu_rolling_outlier"],
        "velocidad": [90.0, np.nan],
        "ax": [np.nan, 120.0],
        "ay": [np.nan, 180.0],
        "az": [np.nan, 95.0],
    }
)
summary_sample = {
    "source_key": "TEST_SOURCE",
    "filter_steps": [
        {"source_key": "TEST_SOURCE", "stage": "gps", "filter_name": "gps_invalid_speed", "before_rows": 100, "dropped_rows": 1, "after_rows": 99},
        {"source_key": "TEST_SOURCE", "stage": "gps", "filter_name": "gps_fix_lt_1", "before_rows": 99, "dropped_rows": 0, "after_rows": 99},
    ]
}
reasons_df = build_discard_reason_explanations(
    dropped_sample,
    summary=summary_sample,
    reason_legend={"gps_invalid_speed": "Velocidad invalida", "gps_fix_lt_1": "Fix invalido"},
)
assert {"reason", "descripcion", "dropped_rows", "aplicada"}.issubset(reasons_df.columns)
assert "gps_fix_lt_1" in set(reasons_df["reason"])
assert int(reasons_df.loc[reasons_df["reason"] == "gps_fix_lt_1", "dropped_rows"].iloc[0]) == 0

# Test 5: build_unified_discard_table unifica columnas clave sin 'aplicada' ni 'dropped_rows_filter_step'
unified_df = build_unified_discard_table(
    dropped_sample,
    summary=summary_sample,
    reason_legend={"gps_invalid_speed": "Velocidad invalida", "gps_fix_lt_1": "Fix invalido"},
)
assert {
    "source_key",
    "stage",
    "reason",
    "descripcion",
    "before_rows",
    "dropped_rows",
    "pct_over_dropped",
    "after_rows",
}.issubset(unified_df.columns)
assert "aplicada" not in unified_df.columns
assert "dropped_rows_filter_step" not in unified_df.columns

# Test 6: build_outlier_discard_stats devuelve criterio y rango observado por razon
stats_df = build_outlier_discard_stats(
    dropped_sample,
    summary=summary_sample,
    reason_legend={"gps_invalid_speed": "Velocidad invalida", "gps_fix_lt_1": "Fix invalido"},
    rule_criteria={"gps_invalid_speed": "velocidad fuera de rango", "gps_fix_lt_1": "fix < 1"},
    reason_columns={"gps_invalid_speed": ["velocidad"], "imu_rolling_outlier": ["ax", "ay", "az"]},
)
assert {"reason", "criterio_descarte", "rango_valores_descartados", "n_descartes", "pct_over_dropped"}.issubset(stats_df.columns)
assert "gps_invalid_speed" in set(stats_df["reason"])

print("Tests basicos OK")

Tests basicos OK


## Notas de tuning de ventanas estadisticas

Para ajustar sensibilidad del filtro rolling IMU en el parser:
- `window_size`: aumenta para suavizar ruido local y detectar anomalias mas globales.
- `iqr_multiplier`: baja este valor para hacer el filtro mas estricto en ventanas con IQR estable.
- `mad_multiplier`: controla fallback robusto cuando IQR ~ 0.
- `min_axes_to_flag`: con valor `2` reduces falsos positivos frente a ruido aislado en un solo eje.

Estos parametros se configuran en `config/config.py` dentro de `OUTLIER_FILTER_CONFIG['imu']`.